In [1]:
import joblib
import pandas as pd
import sys
sys.path.append("..")
from src.risk_scoring import Calc_Risk

X_test = pd.read_csv('../data/selected/X_test_selected.csv')
y_test = pd.read_csv('../data/selected/y_test.csv').squeeze()

lr_model = joblib.load('../models/logistic_regression.pkl')
dt_model = joblib.load('../models/decision_tree.pkl')
rf_model = joblib.load('../models/random_forest.pkl')

print(f'Test shape: {X_test.shape} | Target shape: {y_test.shape}')
assert X_test.shape[1] == 15, 'Feature count mismatch!'
print('Features and models loaded successfully.')

Test shape: (2997, 15) | Target shape: (2997,)
Features and models loaded successfully.


In [2]:
lr_pred, dt_pred, rf_pred = (
    lr_model.predict(X_test),
    dt_model.predict(X_test),
    rf_model.predict(X_test),
)
lr_prob = lr_model.predict_proba(X_test)[:, 1]
dt_prob = dt_model.predict_proba(X_test)[:, 1]
rf_prob = rf_model.predict_proba(X_test)[:, 1]

results = pd.DataFrame(
    {
        'actual': y_test.values,
        'lr_prediction': lr_pred,
        'dt_prediction': dt_pred,
        'rf_prediction': rf_pred,
        'lr_probability': lr_prob,
        'dt_probability': dt_prob,
        'rf_probability': rf_prob,
    }
)

display(results.head())

,actual,lr_prediction,dt_prediction,rf_prediction,lr_probability,dt_probability,rf_probability
0,1,1,1,1,0.995459,1.0,1.00
1,0,0,0,0,0.482308,0.0,0.08
2,1,1,1,1,0.996116,1.0,0.99
3,1,1,1,1,0.998464,1.0,1.00
4,1,1,1,1,0.994614,1.0,1.00


In [3]:
#Agreement/Disagreement rates
results['all_agree'] = (
    (results['lr_prediction'] == results['dt_prediction'])
    & (results['dt_prediction'] == results['rf_prediction'])
)

print(f'Total samples: {len(results)}')
print(f'All three agree: {results["all_agree"].sum()}')
print(f'Agreement rate: {results["all_agree"].mean():.2%}')
print(f'Disagreements: {(~results["all_agree"]).sum()}')

# LR-DT-RF -> our pattern
results['prediction_pattern'] = (
    results['lr_prediction'].astype(str)
    + '-'
    + results['dt_prediction'].astype(str)
    + '-'
    + results['rf_prediction'].astype(str)
)

print('\nPrediction Patterns:')
print(results['prediction_pattern'].value_counts())

Total samples: 2997
All three agree: 2456
Agreement rate: 81.95%
Disagreements: 541

Prediction Patterns:
prediction_pattern
1-1-1    2184
0-0-0     272
1-0-0     209
1-0-1     140
0-1-1      96
1-1-0      47
0-1-0      35
0-0-1      14
Name: count, dtype: int64


In [4]:
disagreements = results[~results['all_agree']].copy()

disagreements['correct_models'] = (
    (disagreements['lr_prediction'] == disagreements['actual']).astype(int)
    + (disagreements['dt_prediction'] == disagreements['actual']).astype(int)
    + (disagreements['rf_prediction'] == disagreements['actual']).astype(int)
)

print('Disagreement Pattern Breakdown:')
print(disagreements['prediction_pattern'].value_counts())

print('\nCorrect model counts in disagreement cases:')
print(disagreements['correct_models'].value_counts().sort_index())

display(disagreements.head(10))

Disagreement Pattern Breakdown:
prediction_pattern
1-0-0    209
1-0-1    140
0-1-1     96
1-1-0     47
0-1-0     35
0-0-1     14
Name: count, dtype: int64

Correct model counts in disagreement cases:
correct_models
1    161
2    380
Name: count, dtype: int64


,actual,lr_prediction,dt_prediction,rf_prediction,lr_probability,dt_probability,rf_probability,all_agree,prediction_pattern,correct_models
9,1,1,0,0,0.718929,0.0,0.042333,False,1-0-0,1
18,0,1,0,0,0.508203,0.0,0.110000,False,1-0-0,2
27,1,1,1,0,0.835362,1.0,0.420000,False,1-1-0,2
31,0,1,0,0,0.637980,0.0,0.170000,False,1-0-0,2
33,1,1,0,0,0.530893,0.0,0.235667,False,1-0-0,1
35,1,1,1,0,0.744030,1.0,0.420000,False,1-1-0,2
36,1,0,1,1,0.370381,1.0,0.890000,False,0-1-1,2
47,1,0,1,1,0.392966,1.0,0.720000,False,0-1-1,2
50,1,0,0,1,0.325116,0.0,0.670000,False,0-0-1,1
54,1,0,1,1,0.379686,1.0,0.710000,False,0-1-1,2


In [5]:
results["lr_risk_score"] = results["lr_probability"] * 100
results["lr_risk_level"] = results["lr_risk_score"].apply(
    lambda x:
        "Very High" if x >= 80 else
        "High" if x >= 60 else
        "Medium" if x >= 40 else
        "Low" if x >= 20 else
        "Very Low"
)
results[
    [
        "actual",
        "lr_prediction",
        "lr_probability",
        "lr_risk_score",
        "lr_risk_level"
    ]
].head(20)
print(results["lr_risk_level"].value_counts())

lr_risk_level
Very High    1898
High          503
Medium        320
Low           203
Very Low       73
Name: count, dtype: int64


In [6]:
risk_vs_actual = pd.crosstab(
    results["lr_risk_level"],
    results["actual"]
)

print(risk_vs_actual)

risk_vs_actual_pct = pd.crosstab(
    results["lr_risk_level"],
    results["actual"],
    normalize="index"
) * 100

print(risk_vs_actual_pct.round(2))



actual           0     1
lr_risk_level           
High           145   358
Low            165    38
Medium         163   157
Very High       85  1813
Very Low        58    15
actual             0      1
lr_risk_level              
High           28.83  71.17
Low            81.28  18.72
Medium         50.94  49.06
Very High       4.48  95.52
Very Low       79.45  20.55


In [7]:
results["lr_probability_bin"] = pd.cut(
    results["lr_probability"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=[
        "0-20%",
        "20-40%",
        "40-60%",
        "60-80%",
        "80-100%"
    ],
    include_lowest=True
)

probability_analysis = results.groupby(
    "lr_probability_bin",
    observed=False
)["actual"].agg(
    samples="count",
    phishing_rate="mean"
)

probability_analysis["phishing_rate"] *= 100

print(probability_analysis.round(2))

                    samples  phishing_rate
lr_probability_bin                        
0-20%                    73          20.55
20-40%                  203          18.72
40-60%                  320          49.06
60-80%                  503          71.17
80-100%                1898          95.52
